# Task 4: Query Performance Benchmark

Tests whether OPTIMIZE + ZORDER BY (carrier_id) makes queries on `logistics.gold.fact_shipment_activity` faster, by comparing query plans and timed runs before and after optimization.

In [0]:
# TASK 4: Query Performance Benchmark

from time import perf_counter
from pyspark.sql import functions as F

TABLE = "logistics.gold.fact_shipment_activity"

# --------------------------------------------------
# 1. Check table
# --------------------------------------------------

if not spark.catalog.tableExists(TABLE):
    raise RuntimeError(
        f"Missing table {TABLE}. Run Tasks 1-3 first."
    )

print(f"Table found: {TABLE}")

In [0]:
# --------------------------------------------------
# 2. Define representative query
# --------------------------------------------------

def benchmark_query():

    return (
        spark.table(TABLE)

        .filter(
            F.col("delivery_time").isNotNull()
        )

        .groupBy("carrier_id")

        .agg(
            F.countDistinct("shipment_id")
            .alias("delivered_shipments"),

            F.round(
                F.avg("delivery_duration_hours"),
                2
            ).alias("avg_lead_time_hours")
        )

        .orderBy("carrier_id")
    )

display(benchmark_query())

In [0]:
# --------------------------------------------------
# 3. Benchmark helper
# --------------------------------------------------

def measure(label, repetitions=3):

    execution_times = []

    for i in range(repetitions):

        start = perf_counter()

        benchmark_query().collect()

        elapsed = perf_counter() - start

        execution_times.append(elapsed)

    mean_time = (
        sum(execution_times)
        / len(execution_times)
    )

    print(
        f"{label}: "
        f"Times={execution_times}, "
        f"Mean={mean_time:.3f} seconds"
    )

    return execution_times

In [0]:
# --------------------------------------------------
# 4. Explain query plan before optimization
# --------------------------------------------------

print("QUERY PLAN BEFORE OPTIMIZATION")

benchmark_query().explain("formatted")

# Warm-up
benchmark_query().count()

In [0]:
# --------------------------------------------------
# 5. Measure before optimization
# --------------------------------------------------

before_times = measure(
    "BEFORE OPTIMIZE"
)

In [0]:
# --------------------------------------------------
# 6. Optimize Delta table
# --------------------------------------------------

display(
    spark.sql(
        f"""
        OPTIMIZE {TABLE}
        ZORDER BY (carrier_id)
        """
    )
)

In [0]:
# --------------------------------------------------
# 7. Explain query plan after optimization
# --------------------------------------------------

print("QUERY PLAN AFTER OPTIMIZATION")

benchmark_query().explain("formatted")

# Warm-up
benchmark_query().count()

In [0]:
# --------------------------------------------------
# 8. Measure after optimization
# --------------------------------------------------

after_times = measure(
    "AFTER OPTIMIZE"
)

In [0]:
# --------------------------------------------------
# 9. Compare results
# --------------------------------------------------

before_mean = sum(before_times) / len(before_times)
after_mean = sum(after_times) / len(after_times)

print("BEFORE:", round(before_mean, 3), "seconds")
print("AFTER:", round(after_mean, 3), "seconds")

if before_mean > 0:
    improvement = (
        (before_mean - after_mean)
        / before_mean
    ) * 100

    print(
        "Measured improvement:",
        round(improvement, 2),
        "%"
    )

print(
    "Report actual measurements. "
    "OPTIMIZE does not guarantee improvement."
)